# Fabric Lineage & Catalog Metadata Hydrator

This notebook calls the Microsoft Fabric REST APIs to extract metadata about all items
in your tenant, then writes the results as Delta tables in your Lakehouse.

A **Fabric Data Agent** can then query these tables to answer natural-language questions like:
- *When was data X last updated?*
- *Who owns the sales lakehouse?*
- *What items are in the marketing workspace?*
- *Who has access to workspace Y?*

### Setup
1. Attach this notebook to a Lakehouse
2. Run all cells
3. Schedule via a Fabric Pipeline for auto-refresh


## Configuration

In [ ]:
TABLE_WORKSPACES = 'lineage_workspaces'
TABLE_ITEMS = 'lineage_items'
TABLE_REFRESH_HISTORY = 'lineage_refresh_history'
TABLE_ACCESS = 'lineage_workspace_access'
TABLE_LINEAGE = 'lineage_item_dependencies'
TABLE_SCAN_LOG = 'lineage_scan_log'

FABRIC_API_BASE = 'https://api.fabric.microsoft.com/v1'
ADMIN_API_BASE = 'https://api.fabric.microsoft.com/v1/admin'

SCAN_DATASET_SCHEMA = True
SCAN_DATASOURCE_DETAILS = True

print('Configuration loaded.')


## Authentication Helpers

In [ ]:
import requests
import json
from datetime import datetime, timezone

def get_fabric_token():
    return mssparkutils.credentials.getToken('https://api.fabric.microsoft.com')

def fabric_get(url, params=None):
    headers = {'Authorization': f'Bearer {get_fabric_token()}', 'Content-Type': 'application/json'}
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def fabric_post(url, body=None):
    headers = {'Authorization': f'Bearer {get_fabric_token()}', 'Content-Type': 'application/json'}
    resp = requests.post(url, headers=headers, json=body, timeout=30)
    resp.raise_for_status()
    return resp.json() if resp.content else {}

def paginate_get(url, key='value', params=None):
    all_items = []
    next_url = url
    while next_url:
        data = fabric_get(next_url, params=params)
        all_items.extend(data.get(key, []))
        next_url = data.get('continuationUri') or data.get('@odata.nextLink')
        params = None
    return all_items

print('Auth helpers ready.')


## Fetch Workspaces

In [ ]:
def fetch_workspaces():
    workspaces = paginate_get(f'{FABRIC_API_BASE}/workspaces')
    rows = []
    for ws in workspaces:
        rows.append({
            'workspace_id': ws.get('id', ''),
            'workspace_name': ws.get('displayName', ''),
            'description': ws.get('description', ''),
            'type': ws.get('type', ''),
            'state': ws.get('state', ''),
            'capacity_id': ws.get('capacityId', ''),
            'scanned_at': datetime.now(timezone.utc).isoformat(),
        })
    print(f'Fetched {len(rows)} workspaces.')
    return rows

workspace_rows = fetch_workspaces()


## Fetch Items per Workspace

In [ ]:
def fetch_items(workspaces):
    all_items = []
    for ws in workspaces:
        ws_id = ws['workspace_id']
        ws_name = ws['workspace_name']
        try:
            items = paginate_get(f'{FABRIC_API_BASE}/workspaces/{ws_id}/items')
            for item in items:
                all_items.append({
                    'workspace_id': ws_id,
                    'workspace_name': ws_name,
                    'item_id': item.get('id', ''),
                    'item_name': item.get('displayName', ''),
                    'item_type': item.get('type', ''),
                    'description': item.get('description', ''),
                    'scanned_at': datetime.now(timezone.utc).isoformat(),
                })
        except Exception as e:
            print(f'  Warning: could not fetch items for workspace {ws_name}: {e}')
    print(f'Fetched {len(all_items)} items across {len(workspaces)} workspaces.')
    return all_items

item_rows = fetch_items(workspace_rows)


## Fetch Workspace Role Assignments (Access)

In [ ]:
def fetch_workspace_access(workspaces):
    all_access = []
    for ws in workspaces:
        ws_id = ws['workspace_id']
        ws_name = ws['workspace_name']
        try:
            assignments = paginate_get(f'{FABRIC_API_BASE}/workspaces/{ws_id}/roleAssignments')
            for ra in assignments:
                principal = ra.get('principal', {})
                all_access.append({
                    'workspace_id': ws_id,
                    'workspace_name': ws_name,
                    'principal_id': principal.get('id', ''),
                    'principal_name': principal.get('displayName', ''),
                    'principal_type': principal.get('type', ''),
                    'role': ra.get('role', ''),
                    'scanned_at': datetime.now(timezone.utc).isoformat(),
                })
        except Exception as e:
            print(f'  Warning: could not fetch access for workspace {ws_name}: {e}')
    print(f'Fetched {len(all_access)} role assignments.')
    return all_access

access_rows = fetch_workspace_access(workspace_rows)


## Scanner API (Rich Metadata + Lineage)

> **Note:** Requires Fabric Admin permissions. If unavailable, this cell will print a warning and remaining cells still work.


In [ ]:
import time

def run_scanner(workspaces):
    ws_ids = [ws['workspace_id'] for ws in workspaces]
    print('Triggering metadata scan...')
    scan_body = {'workspaces': ws_ids, 'datasetSchema': SCAN_DATASET_SCHEMA, 'datasourceDetails': SCAN_DATASOURCE_DETAILS}
    try:
        scan_response = fabric_post(f'{ADMIN_API_BASE}/workspaces/getInfo', body=scan_body)
    except requests.exceptions.HTTPError as e:
        print(f'Scanner API error: {e}')
        print('Note: Scanner APIs require Fabric Admin permissions.')
        return [], []
    scan_id = scan_response.get('id')
    if not scan_id:
        print('No scan ID returned. Check admin permissions.')
        return [], []
    print(f'Scan ID: {scan_id} -- waiting for completion...')
    for _ in range(30):
        status = fabric_get(f'{ADMIN_API_BASE}/workspaces/scanStatus/{scan_id}')
        if status.get('status') == 'Succeeded':
            print('Scan completed.')
            break
        time.sleep(5)
    else:
        print('Scan timed out after 150 seconds.')
        return [], []
    result = fabric_get(f'{ADMIN_API_BASE}/workspaces/scanResult/{scan_id}')
    refresh_rows = []
    dependency_rows = []
    for ws in result.get('workspaces', []):
        ws_id = ws.get('id', '')
        ws_name = ws.get('name', '')
        for dataset in ws.get('datasets', []):
            refresh_rows.append({
                'workspace_id': ws_id, 'workspace_name': ws_name,
                'item_id': dataset.get('id', ''), 'item_name': dataset.get('name', ''),
                'item_type': 'SemanticModel',
                'configured_by': dataset.get('configuredBy', ''),
                'is_refreshable': str(dataset.get('isRefreshable', '')),
                'last_refresh_time': dataset.get('refreshedDate', ''),
                'sensitivity_label': dataset.get('sensitivityLabel', {}).get('labelId', ''),
                'endorsement': dataset.get('endorsementDetails', {}).get('endorsement', ''),
                'scanned_at': datetime.now(timezone.utc).isoformat(),
            })
        for item_type_key in ['reports', 'dashboards', 'dataflows', 'datamarts']:
            for item in ws.get(item_type_key, []):
                refresh_rows.append({
                    'workspace_id': ws_id, 'workspace_name': ws_name,
                    'item_id': item.get('id', ''), 'item_name': item.get('name', ''),
                    'item_type': item_type_key.rstrip('s').title(),
                    'configured_by': item.get('configuredBy', item.get('createdBy', '')),
                    'is_refreshable': '',
                    'last_refresh_time': item.get('modifiedDateTime', item.get('createdDateTime', '')),
                    'sensitivity_label': item.get('sensitivityLabel', {}).get('labelId', ''),
                    'endorsement': item.get('endorsementDetails', {}).get('endorsement', ''),
                    'scanned_at': datetime.now(timezone.utc).isoformat(),
                })
        for dataset in ws.get('datasets', []):
            for upstream in dataset.get('upstreamDatasets', []):
                dependency_rows.append({
                    'workspace_id': ws_id, 'workspace_name': ws_name,
                    'item_id': dataset.get('id', ''), 'item_name': dataset.get('name', ''),
                    'item_type': 'SemanticModel',
                    'depends_on_id': upstream.get('targetDatasetId', ''),
                    'dependency_type': 'upstreamDataset',
                    'scanned_at': datetime.now(timezone.utc).isoformat(),
                })
            for source in dataset.get('datasourceUsages', []):
                dependency_rows.append({
                    'workspace_id': ws_id, 'workspace_name': ws_name,
                    'item_id': dataset.get('id', ''), 'item_name': dataset.get('name', ''),
                    'item_type': 'SemanticModel',
                    'depends_on_id': source.get('datasourceInstanceId', ''),
                    'dependency_type': 'datasource',
                    'scanned_at': datetime.now(timezone.utc).isoformat(),
                })
    print(f'Scanner extracted {len(refresh_rows)} item details, {len(dependency_rows)} dependencies.')
    return refresh_rows, dependency_rows

refresh_rows, dependency_rows = run_scanner(workspace_rows)


## Write All Tables to Lakehouse

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

def write_to_lakehouse(rows, table_name):
    if not rows:
        print(f'  Skipping {table_name} -- no data.')
        return
    df = spark.createDataFrame(rows)
    df.write.mode('overwrite').format('delta').saveAsTable(table_name)
    print(f'  {table_name}: {len(rows)} rows written.')

print('Writing tables to Lakehouse...')
write_to_lakehouse(workspace_rows, TABLE_WORKSPACES)
write_to_lakehouse(item_rows, TABLE_ITEMS)
write_to_lakehouse(access_rows, TABLE_ACCESS)
write_to_lakehouse(refresh_rows, TABLE_REFRESH_HISTORY)
write_to_lakehouse(dependency_rows, TABLE_LINEAGE)

scan_log = [{
    'scan_timestamp': datetime.now(timezone.utc).isoformat(),
    'workspaces_scanned': len(workspace_rows),
    'items_scanned': len(item_rows),
    'access_entries': len(access_rows),
    'refresh_entries': len(refresh_rows),
    'dependency_entries': len(dependency_rows),
}]
write_to_lakehouse(scan_log, TABLE_SCAN_LOG)

print('\nHydration complete! Tables are ready for your Data Agent.')


## Sample Queries

Uncomment any query below to verify the data.


In [ ]:
# -- When was the sales model last refreshed?
# spark.sql("SELECT item_name, last_refresh_time, configured_by FROM lineage_refresh_history WHERE LOWER(item_name) LIKE '%sales%' ORDER BY last_refresh_time DESC").show(truncate=False)

# -- Who has access to the marketing workspace?
# spark.sql("SELECT principal_name, principal_type, role FROM lineage_workspace_access WHERE LOWER(workspace_name) LIKE '%marketing%'").show(truncate=False)

# -- What items are in the analytics workspace?
# spark.sql("SELECT item_name, item_type FROM lineage_items WHERE LOWER(workspace_name) LIKE '%analytics%' ORDER BY item_type, item_name").show(truncate=False)

# -- Show all items modified in the last 7 days
# spark.sql("SELECT item_name, item_type, workspace_name, last_refresh_time FROM lineage_refresh_history WHERE last_refresh_time >= date_sub(current_date(), 7) ORDER BY last_refresh_time DESC").show(truncate=False)

# -- What are the upstream dependencies for model X?
# spark.sql("SELECT item_name, depends_on_id, dependency_type FROM lineage_item_dependencies WHERE LOWER(item_name) LIKE '%model_name%'").show(truncate=False)

print('Sample queries available -- uncomment to run.')
